# Depth-12 latent probes: final three-panel figure

Use checkpoints **512, 16384, 262144, and 2097152** from the 12-layer, width-256, 16-head run (`bin2_ter16`, `l20.03`). The panels show second-last level-1, third-last level-1, and third-last level-2 latents, matching the original three-panel figure.

The experiment preserves the original grammar, independent fully ternary train/test sets (22,000 / 10,000 examples; seeds 42 / 43), causal positions, and standardized logistic probes (`C=0.1`). It uses the original depth-6 bundle only for these experiment definitions; every plotted curve is fitted using the depth-12 models.

## 1. Compute missing depth-12 checkpoints

Run the next cell from the repository root. It saves `second_third_last_probe_checkpoint_results_depth12.pkl` after each completed checkpoint and reuses completed stages. The original depth-6 results and plots are preserved.


In [ ]:
# Run this cell once; completed checkpoints are reused on subsequent runs.
from pathlib import Path
import runpy
import sys

scripts_dir = Path.cwd() / 'scripts'
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))
runpy.run_path(str(scripts_dir / 'probe_latent_depth12.py'), run_name='__main__')


## 2. Regenerate the final three-panel plot

Once all four checkpoints are saved, run the next cell independently, including after restarting the kernel. It needs only NumPy, Matplotlib, and the depth-12 result bundle. It exports `latent_probe_three_panel_summary_depth12.png` and `.pdf` with embedding plus L1–L12 on the horizontal axis.


In [ ]:
# Standalone: run only this cell after restarting the kernel.
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

# Set DATA_DIR explicitly if the kernel starts outside the notebook directory.
DATA_DIR = Path.cwd()
RESULTS_PATH = DATA_DIR / "second_third_last_probe_checkpoint_results_depth12.pkl"
with RESULTS_PATH.open("rb") as handle:
    plot_data = pickle.load(handle)

final_results = plot_data["results"]
final_layer_labels = plot_data["layer_labels"]
chance_accuracy = plot_data["chance_accuracy"]
print(f"Loaded plot data from {RESULTS_PATH}")

STAGE_STEPS = [512, 16_384, 262_144, 2_097_152]
STAGE_LABELS = ["Stage 1a", "Stage 1b", "Stage 2", "Stage 3"]
STEP_COLORS = {
    512: "tab:purple",
    16_384: "tab:blue",
    262_144: "tab:orange",
    2_097_152: "tab:green",
}
# Paper notation uses root-based depth and one-based node indices:
# level-1 node 7 -> mu^(2)_8; level-1 node 6 -> mu^(2)_7;
# level-2 node 0 -> mu^(1)_1.
PANELS = [
    ("second_last_level1", r"(a) $\mu^{(2)}_8$"),
    ("third_last_level1", r"(b) $\mu^{(2)}_7$"),
    ("third_last_level2", r"(c) $\mu^{(1)}_1$"),
]

if plot_data['architecture']['num_layers'] != 12 or plot_data['architecture']['embedding_dim'] != 256:
    raise ValueError('Expected the depth-12, width-256 probe bundle')
if len(final_layer_labels) != 13:
    raise ValueError('Expected embedding plus 12 layers')
for step in STAGE_STEPS:
    for name, _ in PANELS:
        curve = np.asarray(final_results[step]['probes'][name]['test'])
        if curve.shape != (13,) or not np.all(np.isfinite(curve)):
            raise ValueError(f'Invalid saved curve: checkpoint {step}, probe {name}')

x_depth = np.arange(len(final_layer_labels))
fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.2), sharex=True, sharey=True)

for ax, (probe_name, title) in zip(axes, PANELS):
    for stage, step in enumerate(STAGE_STEPS, start=1):
        accuracy = final_results[step]["probes"][probe_name]["test"]
        ax.plot(
            x_depth, accuracy, "o-", color=STEP_COLORS[step],
            linewidth=2.0, markersize=5, label=STAGE_LABELS[stage - 1],
        )
    ax.axhline(chance_accuracy, color="gray", linestyle=":", linewidth=1.3)
    ax.set(
        xticks=x_depth, xticklabels=final_layer_labels,
        xlabel="Probed layer", ylim=(0, 1.02),
    )
    ax.set_title(title, fontsize=22, pad=12)
    ax.xaxis.label.set_size(16)
    ax.tick_params(axis="y", labelsize=13)
    ax.tick_params(axis="x", labelsize=11, labelrotation=45)
    ax.grid(alpha=0.2)

axes[0].set_ylabel("Held-out probe accuracy", fontsize=16)
axes[0].legend(loc="lower right", frameon=False, fontsize=13)
fig.tight_layout(w_pad=1.6)

PDF_PATH = DATA_DIR / "latent_probe_three_panel_summary_depth12.pdf"
PNG_PATH = DATA_DIR / "latent_probe_three_panel_summary_depth12.png"
fig.savefig(PDF_PATH, bbox_inches="tight")
fig.savefig(PNG_PATH, dpi=300, bbox_inches="tight")
print(f"Saved figure to {PDF_PATH}")
print(f"Saved figure to {PNG_PATH}")
plt.show()


## 3. Compare the deepest latent across depths and all four stages

Plot held-out accuracy for $\mu^{(1)}_1$ against relative transformer layer $k/D$ (embedding at 0). Colours and separate stage/depth legends match the varying-tree panel in `plot_fixed_vs_varying_tree.py`: purple/blue/orange/green for Stages 1a/1b/2/3; dashed open squares for D=6 and solid circles for D=12.

The cached latent-probe checkpoints are **512, 16384, 262144, 1048576** for D=6 and **512, 16384, 262144, 2097152** for D=12. D=6 Stage 1b comes from `second_third_last_probe_step_16384.pkl`; its final checkpoint differs from the token-loss reference plot (2965824). The cell runs independently and exports `latent_probe_depth6_vs_depth12_four_stages.pdf` and `.png`.


In [ ]:
# Standalone comparison of the deepest latent at all four training stages.
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({
    "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

DATA_DIR = Path.cwd()
PROBE_NAME = "third_last_level2"  # Deepest latent: mu^(1)_1.
STAGE_LABELS = ["Stage 1a", "Stage 1b", "Stage 2", "Stage 3"]
STAGE_COLORS = ["tab:purple", "tab:blue", "tab:orange", "tab:green"]
CONFIGS = [
    (6, "second_third_last_probe_checkpoint_results.pkl",
     [512, 16_384, 262_144, 1_048_576]),
    (12, "second_third_last_probe_checkpoint_results_depth12.pkl",
     [512, 16_384, 262_144, 2_097_152]),
]

curves = {}
reference = None
for D, filename, steps in CONFIGS:
    with (DATA_DIR / filename).open("rb") as handle:
        data = pickle.load(handle)
    if data["architecture"]["num_layers"] != D:
        raise ValueError(f"Expected {D} transformer layers in {filename}")
    # The D=6 Stage-1b probes were computed separately from the original bundle.
    results = dict(data["results"])
    if D == 6:
        with (DATA_DIR / "second_third_last_probe_step_16384.pkl").open("rb") as handle:
            extra = pickle.load(handle)
        for key in ("grammar", "dataset", "architecture", "probe_C", "chance_accuracy"):
            if extra[key] != data[key]:
                raise ValueError(f"Stage-1b probe metadata mismatch: {key}")
        if extra["probe_metadata"][PROBE_NAME] != data["probe_metadata"][PROBE_NAME]:
            raise ValueError("Stage-1b latent definition mismatch")
        results[16_384] = extra["results"][16_384]
    if reference is None:
        reference = data
    for key in ("grammar", "dataset", "probe_C", "chance_accuracy"):
        if data[key] != reference[key]:
            raise ValueError(f"Probe experiments differ between depths: {key}")
    if data["probe_metadata"][PROBE_NAME] != reference["probe_metadata"][PROBE_NAME]:
        raise ValueError("Latent definitions differ between depths")
    for stage, step in enumerate(steps):
        result = results[step]
        if result["saved_step"] != step:
            raise ValueError(f"Checkpoint mismatch: D={D}, step={step}")
        accuracy = np.asarray(result["probes"][PROBE_NAME]["test"], dtype=float)
        if (accuracy.shape != (D + 1,) or not np.all(np.isfinite(accuracy))
                or np.any((accuracy < 0) | (accuracy > 1))):
            raise ValueError(f"Invalid accuracy curve: D={D}, step={step}")
        curves[D, stage] = accuracy
    print(f"D={D}: " + ", ".join(f"{label}: {step:,}" for label, step in zip(STAGE_LABELS, steps)))

fig, ax = plt.subplots(figsize=(5.2, 3.65))
for (D, stage), accuracy in curves.items():
    color = STAGE_COLORS[stage]
    ax.plot(np.arange(D + 1) / D, accuracy, color=color,
            ls="--" if D == 6 else "-", marker="s" if D == 6 else "o",
            markerfacecolor="white" if D == 6 else color,
            lw=1.5, ms=2.8, zorder=4)

ax.axhline(reference["chance_accuracy"], color="grey", ls=":", lw=.9)
ax.set(xlabel=r"Relative transformer layer $k/D$",
       ylabel=r"Probe accuracy $\mathcal{A}_{Z}^{(k,t)}$", title=r"Deepest latent $\mu^{(1)}_1$",
       xlim=(-.025, 1.025), ylim=(0, 1.02), xticks=np.linspace(0, 1, 5))
ax.grid(alpha=.14, lw=.5)
stage_handles = [Line2D([0], [0], color=color, lw=1.6, label=label)
                 for color, label in zip(STAGE_COLORS, STAGE_LABELS)]
fig.legend(handles=stage_handles, loc="upper center", bbox_to_anchor=(.55, .995),
           ncol=4, frameon=False, handlelength=1.4, columnspacing=.9)
depth_handles = [Line2D([0], [0], color="black", ls=ls, marker=marker,
                       markerfacecolor="white" if D == 6 else "black",
                       lw=1.4, ms=3, label=f"$D={D}$")
                 for D, ls, marker in [(6, "--", "s"), (12, "-", "o")]]
ax.legend(handles=depth_handles, loc="lower right", fontsize=7,
          frameon=False, ncol=2, handlelength=1.6, columnspacing=1)
fig.subplots_adjust(left=.14, right=.97, bottom=.15, top=.79)

for extension in ("pdf", "png"):
    output = DATA_DIR / f"latent_probe_depth6_vs_depth12_four_stages.{extension}"
    fig.savefig(output, dpi=300)
    print(f"Saved figure to {output}")
plt.show()


In [ ]:
# Standalone: run this cell after a kernel restart; no probe fitting needed.
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

# Run from the notebook directory, or set DATA_DIR to the results directory.
DATA_DIR = Path.cwd()
RESULTS_PATH = DATA_DIR / "second_third_last_probe_checkpoint_results_depth12.pkl"
with RESULTS_PATH.open("rb") as handle:
    plot_data = pickle.load(handle)
if plot_data["architecture"]["num_layers"] != 12:
    raise ValueError("Expected the D=12 probe results")

# All four stages, including t=16384, are already in the D=12 bundle.
STAGE_STEPS = [512, 16_384, 262_144, 2_097_152]
STAGE_LABELS = ["Stage 1a", "Stage 1b", "Stage 2", "Stage 3"]
PROBES = {
    "second_last_level1": (r"$\mu^{(2)}_8$", "tab:purple"),
    "third_last_level1": (r"$\mu^{(2)}_7$", "tab:brown"),
    "third_last_level2": (r"$\mu^{(1)}_1$", "tab:cyan"),
}
# Use L6, halfway through the D=12 transformer.
l6_index = plot_data["layer_labels"].index("L6")
accuracies = {
    name: np.array([
        plot_data["results"][step]["probes"][name]["test"][l6_index]
        for step in STAGE_STEPS
    ], dtype=float)
    for name in PROBES
}

# One half of a ~7-inch double-column figure, allowing a gap for the sketch.
FIGSIZE = (3.4, 2.8)
with plt.rc_context({
    "font.size": 8, "axes.labelsize": 9, "legend.fontsize": 8,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "pdf.fonttype": 42, "ps.fonttype": 42,
}):
    fig, ax = plt.subplots(figsize=FIGSIZE)
    x = np.arange(len(STAGE_STEPS))
    bar_width = 0.23
    for i, (name, (label, color)) in enumerate(PROBES.items()):
        ax.bar(x + (i - 1) * bar_width, accuracies[name],
               width=bar_width, color=color, label=label)

    ax.axhline(plot_data["chance_accuracy"], color="gray",
               linestyle=":", linewidth=1.0)
    ax.set(xticks=x, xticklabels=STAGE_LABELS, ylim=(0, 1.02),
           xlabel="Learning stage", ylabel=r"Probe accuracy $\mathcal{A}_Z^{(k,t)}$")
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.2)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02),
              ncol=3, fontsize=11, columnspacing=1.2, handletextpad=0.5,
              frameon=False, borderaxespad=0)
    # Fixed margins preserve the requested physical dimensions in both exports.
    fig.subplots_adjust(left=0.17, right=0.98, bottom=0.17, top=0.82)
    for extension in ("pdf", "png"):
        output_path = DATA_DIR / f"latent_probe_accuracy_four_stages_depth12.{extension}"
        fig.savefig(output_path, dpi=300)
        print(f"Saved {output_path}")
    plt.show()
